In [36]:
# MU-Glioma-Post data

import os
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Setup paths
segmentation_volumes_path = "/Users/kasunachinthaperera/Documents/Final Year Project:Thesis/Data/PKG - MU-Glioma-Post/MU-Glioma-Post_Segmentation_Volumes.xlsx"
clinical_data_path = "/Users/kasunachinthaperera/Documents/Final Year Project:Thesis/Data/PKG - MU-Glioma-Post/MU-Glioma-Post_ClinicalData-July2025.xlsx"
image_data_path = "/Users/kasunachinthaperera/Documents/Final Year Project:Thesis/Data/PKG - MU-Glioma-Post/MU-Glioma-Post"

# 2. Load Clinical Data
clinical_data = pd.ExcelFile(clinical_data_path)
for sheet in clinical_data.sheet_names:
    df = pd.read_excel(clinical_data_path, sheet_name=sheet)
    print(f"Sheet: {sheet} | Patients: {len(df)} | Columns: {df.columns.tolist()}")

# 3. Get Patient IDs from Image Data
patient_ids = [d for d in os.listdir(image_data_path) 
               if os.path.isdir(os.path.join(image_data_path, d)) and not d.startswith('.')]
patient_ids.sort()

# 4. Iterate through each Patient for timepoint and image verification
for pid in patient_ids:
    patient_folder_path = os.path.join(image_data_path, pid)
    
    # Get Timepoint folders
    timepoints = sorted([t for t in os.listdir(patient_folder_path) 
                        if os.path.isdir(os.path.join(patient_folder_path, t)) and not t.startswith('.')])
    print(f"Patient ID: {pid} | Timepoints: {timepoints}")


# Segmentation Volumes - Read Excel and filter for our patients
segmentation_volumes = pd.ExcelFile(segmentation_volumes_path)
for sheet in segmentation_volumes.sheet_names:
    df = pd.read_excel(segmentation_volumes_path, sheet_name=sheet)
    print(f"Sheet: {sheet} | Patients: {len(df)} | Columns: len(columns)")
patient_column = "Patient ID"

# Read each sheet separately
necrotic_df = pd.read_excel(segmentation_volumes_path, sheet_name="Necrotic Tumor Core (Label1)")
necrotic_filtered = necrotic_df[necrotic_df[patient_column].isin(patient_ids)]

edema_df = pd.read_excel(segmentation_volumes_path, sheet_name="Tumor Infiltration and Edema")
edema_filtered = edema_df[edema_df[patient_column].isin(patient_ids)]

resection_df = pd.read_excel(segmentation_volumes_path, sheet_name="Resection Cavity (Label4)")
resection_filtered = resection_df[resection_df[patient_column].isin(patient_ids)]


Sheet: Data Dictionary | Patients: 184 | Columns: ['Data Collection Name', 'Data Descriptor /Metadata Name']
Sheet: MU Glioma Post | Patients: 203 | Columns: ['Patient ID', 'Sex at Birth', 'Race', 'Age at diagnosis', 'Primary Diagnosis', 'Grade of Primary Brain Tumor', 'Stereotactic Biopsy before Surgical Resection', 'Progression', 'Time to First Progression (Days)', 'Type of 1st Progression', 'Second Progression/Recurrence', 'Type of 2nd Progression', 'Multiple surgeries', 'Hospice', 'Overall Survival (Death)', 'Number of days from Diagnosis to death (Days)', 'IDH1 mutation', 'IDH2 mutation', '1p/19q', 'ATRX mutation', 'MGMT methylation', 'BRAF V600E mutation', 'TERT promoter mutation', 'Chromosome 7 gain and Chromosome 10 loss', 'H3-3A mutation', 'EGFR amplification', 'PTEN mutation', 'CDKN2A/B deletion', 'TP53 alteration', 'Other mutations/alterations', 'Previous Brain Tumor', 'Type of previous brain tumor', 'Year of previous surgery', 'Grade of Previous brain tumor', 'Number of day

In [37]:
# Image and Segmentation Data - Verify existence of NIfTI files for each patient and timepoint
import os
import nibabel as nib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re

# 1. PATHS
segmentation_volume_path = "/Users/kasunachinthaperera/Documents/Final Year Project:Thesis/Data/PKG - MU-Glioma-Post/MU-Glioma-Post_Segmentation_Volumes.xlsx"
image_data_path = "/Users/kasunachinthaperera/Documents/Final Year Project:Thesis/Data/PKG - MU-Glioma-Post/MU-Glioma-Post"

# 2. EXTRACT PATIENT IDS
patient_ids = sorted([d for d in os.listdir(image_data_path) 
                     if os.path.isdir(os.path.join(image_data_path, d)) and not d.startswith('.')])
print(f"Found {len(patient_ids)} patients with Image Data")

# 3. LOAD SEGMENTATION VOLUMES (All Sheets)
segmentation_volumes = pd.ExcelFile(segmentation_volume_path)
volume_data = {}

for sheet in segmentation_volumes.sheet_names:
    # Read the sheet and clean column names
    df = pd.read_excel(segmentation_volume_path, sheet_name=sheet)
    df.columns = df.columns.str.strip()
    
    # Standardize column name to Patient_ID
    if 'Patient ID' in df.columns:
        df = df.rename(columns={'Patient ID': 'Patient_ID'})
        
        # 1. GENERATE TIMEPOINT LABELS
        # Since the ID repeats, cumcount() assigns 0 to the 1st occurrence, 1 to the 2nd, etc.
        # We add 1 and convert to "Timepoint_X" to match your image folder names.
        df['Timepoint_Index'] = df.groupby('Patient_ID').cumcount() + 1
        df['Timepoint'] = "Timepoint_" + df['Timepoint_Index'].astype(str)
        
        # 2. Filter for only the patients found in your image folder (patient_ids list)
        df_filtered = df[df['Patient_ID'].isin(patient_ids)].copy()
        
        # 3. Store in the dictionary
        # We keep Patient_ID as the index, but Timepoint is now an explicit column
        volume_data[sheet] = df_filtered.set_index('Patient_ID')
        
        print(f"{sheet}: Found {len(df_filtered)} total scans for {df_filtered['Patient_ID'].nunique()} unique patients.")

Found 203 patients with Image Data
Necrotic Tumor Core (Label1): Found 272 total scans for 119 unique patients.
Tumor Infiltration and Edema: Found 419 total scans for 145 unique patients.
Enhancing Tumor Core (Label3): Found 408 total scans for 144 unique patients.
Resection Cavity (Label4): Found 368 total scans for 135 unique patients.


In [50]:
# Clinical Data - Load and set index to Patient ID
clinical_data_path = "/Users/kasunachinthaperera/Documents/Final Year Project:Thesis/Data/PKG - MU-Glioma-Post/MU-Glioma-Post_ClinicalData-July2025.xlsx"
clinical_reader = pd.ExcelFile(clinical_data_path)

# Load the specific sheet and SET THE INDEX to Patient ID
clin_key = clinical_reader.sheet_names[1] 
clinical_df = pd.read_excel(clinical_data_path, sheet_name=clin_key)
clinical_df.columns = clinical_df.columns.str.strip() 
clinical_df = clinical_df.set_index('Patient ID') 

patient_objects = {}

for pid in patient_ids:
    patient_obj = {
        'Patient_ID': pid,
        'biomarkers': {},         
        'treatment': {},          
        'temporal_alignment': {}, 
        'trajectories': {},       
        'outcomes': {},           
        'timepoints': []
    }
    
    # A. Extract ALL VHT Features from clinical_df
    # FIX: Use clinical_df instead of clinical_data[clin_key]
    if pid in clinical_df.index:
        clin_row = clinical_df.loc[pid]
        
        # Handle cases where a PID might appear twice in the Excel
        if isinstance(clin_row, pd.DataFrame):
            clin_row = clin_row.iloc[0]
        
        # 1. BIOMARKERS (Priors)
        patient_obj['biomarkers'] = {
            'Sex': clin_row.get('Sex at Birth'),
            'Age_at_Diagnosis': clin_row.get('Age at diagnosis'),
            'Primary_Diagnosis': clin_row.get('Primary Diagnosis'),
            'Tumor_Grade': clin_row.get('Grade of Primary Brain Tumor'),
            'IDH1': clin_row.get('IDH1 mutation'),
            'IDH2': clin_row.get('IDH2 mutation'),
            '1p19q': clin_row.get('1p/19q'),
            'MGMT': clin_row.get('MGMT methylation'),
            'ATRX': clin_row.get('ATRX mutation'),
            'TERT': clin_row.get('TERT promoter mutation'),
            'EGFR': clin_row.get('EGFR amplification'),
            'PTEN': clin_row.get('PTEN mutation'),
            'CDKN2A_B': clin_row.get('CDKN2A/B deletion'),
            'TP53': clin_row.get('TP53 alteration')
        }
        
        # 2. TREATMENT (Mechanistic Inputs)
        # Note: Added leading spaces to some keys to match the Excel's hidden formatting
        patient_obj['treatment'] = {
            'First_Surgery_Days': clin_row.get('Number of days from Diagnosis to First surgery or procedure'),
            'Multiple_Surgeries': clin_row.get('Multiple surgeries'),
            'Radiation': clin_row.get('Radiation Therapy'),
            'RT_Start_Days': clin_row.get('Number of days from Diagnosis to Radiation Therapy Start date'),
            'RT_End_Days': clin_row.get('Number of days from Diagnosis to Radiation Therapy end date'),
            'RT_Dose_Gy': clin_row.get('Dose'),
            'RT_Fractions': clin_row.get('Number of Fractions'),
            'Chemo': clin_row.get('Initial Chemo Therapy'),
            'Chemo_Name': clin_row.get('Name of Initial Chemo Therapy'),
            'Chemo_Start_Days': clin_row.get(' Number of days from Diagnosis to Initial Chemo Therapy Start date'),
            'Chemo_End_Days': clin_row.get(' Number of days from Diagnosis to Initial Chemo Therapy end date'),
            'Additional_Therapy': clin_row.get('Additional Therapy')
        }
        
        # 3. TEMPORAL ALIGNMENT (Days from Diagnosis per MRI scan)
        patient_obj['temporal_alignment'] = {
            'Timepoint_1': clin_row.get('Number of Days from Diagnosis to 1st MRI (Timepoint_1)'),
            'Timepoint_2': clin_row.get('Number of Days from Diagnosis to 2nd MRI (Timepoint_2)'),
            'Timepoint_3': clin_row.get('Number of Days from Diagnosis to 3rd MRI (Timepoint_3)'),
            'Timepoint_4': clin_row.get('Number of Days from Diagnosis to 4th MRI (Timepoint_4)'),
            'Timepoint_5': clin_row.get('Number of Days from Diagnosis to 5th MRI (Timepoint_5)'),
            'Timepoint_6': clin_row.get('Number of Days from Diagnosis to 6th MRI (Timepoint_6)')
        }

        # 4. OUTCOMES (Validation Data)
        patient_obj['outcomes'] = {
            'Progression': clin_row.get('Progression'),
            'Time_to_Progression': clin_row.get('Time to First Progression (Days)'),
            'OS_Status': clin_row.get('Overall Survival (Death)'),
            'Days_to_Death': clin_row.get('Number of days from Diagnosis to death (Days)'),
            'Hospice': clin_row.get('Hospice')
        }
    
    # B. Map Image structure + timepoints (from folder names)
    patient_folder = Path(image_data_path) / pid
    if patient_folder.exists():
        found_tps = sorted([t.name for t in patient_folder.iterdir() if t.is_dir() and not t.name.startswith('.')])
        patient_obj['timepoints'] = found_tps
    
    # C. Map Longitudinal Volumes using Generated Timepoints
    for tp in patient_obj['timepoints']:
        tp_volumes = {}
        for sheet_name, df in volume_data.items():
            if pid in df.index:
                patient_rows = df.loc[[pid]] 
                match = patient_rows[patient_rows['Timepoint'] == tp]
                
                if not match.empty:
                    # Capture the volume column dynamically
                    vol_cols = [c for c in match.columns if 'Volume' in c or 'Vol' in c]
                    if vol_cols:
                        tp_volumes[sheet_name] = match.iloc[0][vol_cols[0]]
        
        if tp_volumes:
            # Mechanistic total: Necrotic + Edema + Enhancing (excludes Resection Label 4)
            core_vals = [v for k, v in tp_volumes.items() if 'Label4' not in k and 'Resection' not in k]
            tp_volumes['Total_Tumor_Volume'] = sum(core_vals)
            patient_obj['trajectories'][tp] = tp_volumes

    patient_objects[pid] = patient_obj

print(f"Created {len(patient_objects)} Patient Objects with the full {len(list(patient_obj['biomarkers'].keys()) + list(patient_obj['treatment'].keys()) + list(patient_obj['temporal_alignment'].keys()) + list(patient_obj['outcomes'].keys()) + list(patient_obj['trajectories'].keys()))}-feature VHT mapping.")

Created 203 Patient Objects with the full 37-feature VHT mapping.


In [51]:
# =============================================================================
# 4. UNIFIED PATIENT OBJECT INTEGRATION (CLINICAL + IMAGES + VOLUMES)
# =============================================================================
patient_objects = {}

for pid in patient_ids:
    # Initialize the structured object
    patient_obj = {
        'Patient_ID': pid,
        'biomarkers': {},         # 🧬 Biological/Molecular Priors
        'treatment': {},          # ⚡ Mechanistic Perturbations
        'temporal_alignment': {}, # ⏱️ Time (t) for Neural ODEs
        'outcomes': {},           # 🎯 Ground Truth / Prediction
        'timepoints': [],        # 📅 Found in Folders
        'image_paths': {},       # 📂 NIfTI File Names
        'trajectories': {}        # 📏 Volumetric States
    }
    
    # --- PART A: INTEGRATE CLINICAL DATA ---
    if pid in clinical_df.index:
        clin_row = clinical_df.loc[pid]
        if isinstance(clin_row, pd.DataFrame):
            clin_row = clin_row.iloc[0]
        
        # 1. BIOMARKERS (Priors)
        patient_obj['biomarkers'] = {
            'Sex': clin_row.get('Sex at Birth'),
            'Age_at_Diagnosis': clin_row.get('Age at diagnosis'),
            'Primary_Diagnosis': clin_row.get('Primary Diagnosis'),
            'Tumor_Grade': clin_row.get('Grade of Primary Brain Tumor'),
            'IDH1': clin_row.get('IDH1 mutation'),
            'IDH2': clin_row.get('IDH2 mutation'),
            '1p19q': clin_row.get('1p/19q'),
            'MGMT': clin_row.get('MGMT methylation'),
            'ATRX': clin_row.get('ATRX mutation'),
            'TERT': clin_row.get('TERT promoter mutation'),
            'EGFR': clin_row.get('EGFR amplification'),
            'PTEN': clin_row.get('PTEN mutation'),
            'CDKN2A_B': clin_row.get('CDKN2A/B deletion'),
            'TP53': clin_row.get('TP53 alteration')
        }
        
        # 2. TREATMENT (Mechanistic Inputs)
        # Note: Added leading spaces to some keys to match the Excel's hidden formatting
        patient_obj['treatment'] = {
            'First_Surgery_Days': clin_row.get('Number of days from Diagnosis to First surgery or procedure'),
            'Multiple_Surgeries': clin_row.get('Multiple surgeries'),
            'Radiation': clin_row.get('Radiation Therapy'),
            'RT_Start_Days': clin_row.get('Number of days from Diagnosis to Radiation Therapy Start date'),
            'RT_End_Days': clin_row.get('Number of days from Diagnosis to Radiation Therapy end date'),
            'RT_Dose_Gy': clin_row.get('Dose'),
            'RT_Fractions': clin_row.get('Number of Fractions'),
            'Chemo': clin_row.get('Initial Chemo Therapy'),
            'Chemo_Name': clin_row.get('Name of Initial Chemo Therapy'),
            'Chemo_Start_Days': clin_row.get(' Number of days from Diagnosis to Initial Chemo Therapy Start date'),
            'Chemo_End_Days': clin_row.get(' Number of days from Diagnosis to Initial Chemo Therapy end date'),
            'Additional_Therapy': clin_row.get('Additional Therapy')
        }
        
        # 3. TEMPORAL ALIGNMENT (Days from Diagnosis per MRI scan)
        patient_obj['temporal_alignment'] = {
            'Timepoint_1': clin_row.get('Number of Days from Diagnosis to 1st MRI (Timepoint_1)'),
            'Timepoint_2': clin_row.get('Number of Days from Diagnosis to 2nd MRI (Timepoint_2)'),
            'Timepoint_3': clin_row.get('Number of Days from Diagnosis to 3rd MRI (Timepoint_3)'),
            'Timepoint_4': clin_row.get('Number of Days from Diagnosis to 4th MRI (Timepoint_4)'),
            'Timepoint_5': clin_row.get('Number of Days from Diagnosis to 5th MRI (Timepoint_5)'),
            'Timepoint_6': clin_row.get('Number of Days from Diagnosis to 6th MRI (Timepoint_6)')
        }

        # 4. OUTCOMES (Validation Data)
        patient_obj['outcomes'] = {
            'Progression': clin_row.get('Progression'),
            'Time_to_Progression': clin_row.get('Time to First Progression (Days)'),
            'OS_Status': clin_row.get('Overall Survival (Death)'),
            'Days_to_Death': clin_row.get('Number of days from Diagnosis to death (Days)'),
            'Hospice': clin_row.get('Hospice')
        }

    # --- PART B: INTEGRATE IMAGE STRUCTURE ---
    patient_folder = Path(image_data_path) / pid
    if patient_folder.exists():
        found_tps = sorted([t.name for t in patient_folder.iterdir() if t.is_dir() and not t.name.startswith('.')])
        patient_obj['timepoints'] = found_tps
        
        for tp in found_tps:
            # Map NIfTI files
            tp_path = patient_folder / tp
            nifti_files = list(tp_path.glob("*.nii.gz"))
            patient_obj['image_paths'][tp] = [f.name for f in nifti_files]

            # --- PART C: INTEGRATE SEGMENTATION VOLUMES ---
            tp_volumes = {}
            for sheet_name, df_vol in volume_data.items():
                if pid in df_vol.index:
                    patient_rows = df_vol.loc[[pid]]
                    match = patient_rows[patient_rows['Timepoint'] == tp]
                    if not match.empty:
                        vol_cols = [c for c in match.columns if 'Volume' in c or 'Vol' in c]
                        if vol_cols:
                            tp_volumes[sheet_name] = match.iloc[0][vol_cols[0]]
            
            if tp_volumes:
                # Calculate Total Biological Tumor Volume
                core_vals = [v for k, v in tp_volumes.items() if 'Label4' not in k and 'Resection' not in k]
                tp_volumes['Total_Tumor_Volume'] = sum(core_vals)
                patient_obj['trajectories'][tp] = tp_volumes

    patient_objects[pid] = patient_obj

print(f"✅ Integrated {len(patient_objects)} complete patient objects.")

✅ Integrated 203 complete patient objects.


In [54]:
def sanitize_vht_data(obj):
    """Recursively converts NumPy types to standard Python types for clean output."""
    if isinstance(obj, dict):
        return {k: sanitize_vht_data(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [sanitize_vht_data(i) for i in obj]
    elif isinstance(obj, (np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.float64, np.float32)):
        # Convert NaN to None for better readability in outcomes
        return float(obj) if not np.isnan(obj) else None
    else:
        return obj

# Apply the fix to your dictionary
for pid in patient_objects:
    patient_objects[pid] = sanitize_vht_data(patient_objects[pid])

print("✅ Data types standardized to Python primitives.")

for pid, data in patient_objects.items():
    print(f"\nPatient ID: {pid}")
    print(f"Biomarkers: {data['biomarkers']}")
    print(f"Treatment: {data['treatment']}")
    print(f"Temporal Alignment: {data['temporal_alignment']}")
    print(f"Outcomes: {data['outcomes']}")
    print(f"Timepoints: {data['timepoints']}")
    print(f"Image Paths: {data['image_paths']}")
    print(f"Trajectories: {data['trajectories']}")

✅ Data types standardized to Python primitives.

Patient ID: PatientID_0003
Biomarkers: {'Sex': 'Female', 'Age_at_Diagnosis': 57, 'Primary_Diagnosis': 'GBM', 'Tumor_Grade': 4, 'IDH1': 0, 'IDH2': 0, '1p19q': 0, 'MGMT': 4, 'ATRX': 4, 'TERT': 2, 'EGFR': 2, 'PTEN': 0, 'CDKN2A_B': 0, 'TP53': 0}
Treatment: {'First_Surgery_Days': -5, 'Multiple_Surgeries': 1, 'Radiation': 'Yes', 'RT_Start_Days': 17.0, 'RT_End_Days': 65.0, 'RT_Dose_Gy': '60 Gy', 'RT_Fractions': 30.0, 'Chemo': 'Yes', 'Chemo_Name': 'Temozolomide', 'Chemo_Start_Days': None, 'Chemo_End_Days': None, 'Additional_Therapy': 'Temozolomide'}
Temporal Alignment: {'Timepoint_1': 90.0, 'Timepoint_2': 146.0, 'Timepoint_3': None, 'Timepoint_4': None, 'Timepoint_5': 286.0, 'Timepoint_6': None}
Outcomes: {'Progression': 1, 'Time_to_Progression': 286.0, 'OS_Status': 0, 'Days_to_Death': None, 'Hospice': 1}
Timepoints: ['Timepoint_1', 'Timepoint_2', 'Timepoint_5']
Image Paths: {'Timepoint_1': ['PatientID_0003_Timepoint_1_brain_t1c.nii.gz', 'Patien

In [40]:
# =============================================================================
# 5. DETAILED DATA INSPECTION LOOP (REPORTS ALL DATA)
# =============================================================================
print("\n🔍 FULL VHT PATIENT DATA AUDIT")
print("="*100)

for pid, obj in patient_objects.items():
    print(f"\n🆔 PATIENT ID: {pid}")
    
    # Show key biomarkers
    b = obj['biomarkers']
    print(f"🧬 Profile: {b.get('Sex')}, Age {b.get('Age_at_Diagnosis')} | IDH1: {b.get('IDH1')} | MGMT: {b.get('MGMT')}")
    
    if not obj['timepoints']:
        print("    ⚠️ No longitudinal folders found.")
    else:
        for tp in obj['timepoints']:
            days = obj['temporal_alignment'].get(tp, "N/A")
            print(f"\n    ⏱️ {tp} (Day {days}):")
            
            # Show Files
            files = obj['image_paths'].get(tp, [])
            print(f"      📂 Images: {', '.join(files) if files else 'None'}")
            
            # Show Volumes
            vols = obj['trajectories'].get(tp, {})
            if vols:
                v_list = [f"{k}: {v:,.1f}" for k, v in vols.items()]
                print(f"      📏 Volumes: { ' | '.join(v_list) }")
            else:
                print("      📏 Volumes: No segmentation data mapped.")

print("\n" + "="*100)


🔍 FULL VHT PATIENT DATA AUDIT

🆔 PATIENT ID: PatientID_0003
🧬 Profile: Female, Age 57 | IDH1: 0 | MGMT: 4

    ⏱️ Timepoint_1 (Day 90.0):
      📂 Images: PatientID_0003_Timepoint_1_brain_t1c.nii.gz, PatientID_0003_Timepoint_1_brain_t2w.nii.gz, PatientID_0003_Timepoint_1_brain_t1n.nii.gz, PatientID_0003_Timepoint_1_tumorMask.nii.gz, PatientID_0003_Timepoint_1_brain_t2f.nii.gz
      📏 Volumes: Necrotic Tumor Core (Label1): 6,510.0 | Tumor Infiltration and Edema: 44,251.0 | Enhancing Tumor Core (Label3): 33,779.0 | Total_Tumor_Volume: 84,540.0

    ⏱️ Timepoint_2 (Day 146.0):
      📂 Images: PatientID_0003_Timepoint_2_tumorMask.nii.gz, PatientID_0003_Timepoint_2_brain_t1n.nii.gz, PatientID_0003_Timepoint_2_brain_t2f.nii.gz, PatientID_0003_Timepoint_2_brain_t2w.nii.gz, PatientID_0003_Timepoint_2_brain_t1c.nii.gz
      📏 Volumes: Necrotic Tumor Core (Label1): 2,813.0 | Tumor Infiltration and Edema: 45,757.0 | Enhancing Tumor Core (Label3): 37,248.0 | Total_Tumor_Volume: 85,818.0

    ⏱️ Ti

In [ ]:
# 6. CREATE UNIFIED PATIENT OBJECTS
clinical_data_path = "/Users/kasunachinthaperera/Documents/Final Year Project:Thesis/Data/PKG - MU-Glioma-Post/MU-Glioma-Post_ClinicalData-July2025.xlsx"
clinical_reader = pd.ExcelFile(clinical_data_path)

# Load the specific sheet and SET THE INDEX to Patient ID
clin_key = clinical_reader.sheet_names[1] 
clinical_df = pd.read_excel(clinical_data_path, sheet_name=clin_key)
clinical_df.columns = clinical_df.columns.str.strip() 
clinical_df = clinical_df.set_index('Patient ID') # <--- THIS IS CRUCIAL

patient_objects = {}

for pid in patient_ids:
    patient_obj = {
        'Patient_ID': pid,
        'biomarkers': {},         
        'treatment': {},          
        'temporal_alignment': {}, 
        'trajectories': {},       
        'outcomes': {},           
        'timepoints': []
    }
    
    # A. Extract ALL VHT Features from clinical_df
    # FIX: Use clinical_df instead of clinical_data[clin_key]
    if pid in clinical_df.index:
        clin_row = clinical_df.loc[pid]
        
        # Handle cases where a PID might appear twice in the Excel
        if isinstance(clin_row, pd.DataFrame):
            clin_row = clin_row.iloc[0]
        
        # 1. BIOMARKERS (Priors)
        patient_obj['biomarkers'] = {
            'Sex': clin_row.get('Sex at Birth'),
            'Age_at_Diagnosis': clin_row.get('Age at diagnosis'),
            'Primary_Diagnosis': clin_row.get('Primary Diagnosis'),
            'Tumor_Grade': clin_row.get('Grade of Primary Brain Tumor'),
            'IDH1': clin_row.get('IDH1 mutation'),
            'IDH2': clin_row.get('IDH2 mutation'),
            '1p19q': clin_row.get('1p/19q'),
            'MGMT': clin_row.get('MGMT methylation'),
            'ATRX': clin_row.get('ATRX mutation'),
            'TERT': clin_row.get('TERT promoter mutation'),
            'EGFR': clin_row.get('EGFR amplification'),
            'PTEN': clin_row.get('PTEN mutation'),
            'CDKN2A_B': clin_row.get('CDKN2A/B deletion'),
            'TP53': clin_row.get('TP53 alteration')
        }
        
        # 2. TREATMENT (Mechanistic Inputs)
        # Note: Added leading spaces to some keys to match the Excel's hidden formatting
        patient_obj['treatment'] = {
            'First_Surgery_Days': clin_row.get('Number of days from Diagnosis to First surgery or procedure'),
            'Multiple_Surgeries': clin_row.get('Multiple surgeries'),
            'Radiation': clin_row.get('Radiation Therapy'),
            'RT_Start_Days': clin_row.get('Number of days from Diagnosis to Radiation Therapy Start date'),
            'RT_End_Days': clin_row.get('Number of days from Diagnosis to Radiation Therapy end date'),
            'RT_Dose_Gy': clin_row.get('Dose'),
            'RT_Fractions': clin_row.get('Number of Fractions'),
            'Chemo': clin_row.get('Initial Chemo Therapy'),
            'Chemo_Name': clin_row.get('Name of Initial Chemo Therapy'),
            'Chemo_Start_Days': clin_row.get(' Number of days from Diagnosis to Initial Chemo Therapy Start date'),
            'Chemo_End_Days': clin_row.get(' Number of days from Diagnosis to Initial Chemo Therapy end date'),
            'Additional_Therapy': clin_row.get('Additional Therapy')
        }
        
        # 3. TEMPORAL ALIGNMENT (Days from Diagnosis per MRI scan)
        patient_obj['temporal_alignment'] = {
            'Timepoint_1': clin_row.get('Number of Days from Diagnosis to 1st MRI (Timepoint_1)'),
            'Timepoint_2': clin_row.get('Number of Days from Diagnosis to 2nd MRI (Timepoint_2)'),
            'Timepoint_3': clin_row.get('Number of Days from Diagnosis to 3rd MRI (Timepoint_3)'),
            'Timepoint_4': clin_row.get('Number of Days from Diagnosis to 4th MRI (Timepoint_4)'),
            'Timepoint_5': clin_row.get('Number of Days from Diagnosis to 5th MRI (Timepoint_5)'),
            'Timepoint_6': clin_row.get('Number of Days from Diagnosis to 6th MRI (Timepoint_6)')
        }

        # 4. OUTCOMES (Validation Data)
        patient_obj['outcomes'] = {
            'Progression': clin_row.get('Progression'),
            'Time_to_Progression': clin_row.get('Time to First Progression (Days)'),
            'OS_Status': clin_row.get('Overall Survival (Death)'),
            'Days_to_Death': clin_row.get('Number of days from Diagnosis to death (Days)'),
            'Hospice': clin_row.get('Hospice')
        }
    
    # B. Map Image structure + timepoints (from folder names)
    patient_folder = Path(image_data_path) / pid
    if patient_folder.exists():
        found_tps = sorted([t.name for t in patient_folder.iterdir() if t.is_dir() and not t.name.startswith('.')])
        patient_obj['timepoints'] = found_tps
    
    # C. Map Longitudinal Volumes using Generated Timepoints
    for tp in patient_obj['timepoints']:
        tp_volumes = {}
        for sheet_name, df in volume_data.items():
            if pid in df.index:
                patient_rows = df.loc[[pid]] 
                match = patient_rows[patient_rows['Timepoint'] == tp]
                
                if not match.empty:
                    # Capture the volume column dynamically
                    vol_cols = [c for c in match.columns if 'Volume' in c or 'Vol' in c]
                    if vol_cols:
                        tp_volumes[sheet_name] = match.iloc[0][vol_cols[0]]
        
        if tp_volumes:
            # Mechanistic total: Necrotic + Edema + Enhancing (excludes Resection Label 4)
            core_vals = [v for k, v in tp_volumes.items() if 'Label4' not in k and 'Resection' not in k]
            tp_volumes['Total_Tumor_Volume'] = sum(core_vals)
            patient_obj['trajectories'][tp] = tp_volumes

    patient_objects[pid] = patient_obj

print(f"Created {len(patient_objects)} Patient Objects with the full 44-feature VHT mapping.")

Created 203 Patient Objects with the full 44-feature VHT mapping.


In [ ]:
# 7. GENERATE VHT MASTER DATASET & TEMPORAL TRAJECTORIES
print("\n📊 FLATTENING DATA FOR VHT MODELING...")

master_rows = []
longitudinal_rows = []

for pid, obj in patient_objects.items():
    # --- A. Create Master Row (Static Features for Priors & Outcomes) ---
    row = {'Patient_ID': pid}
    
    # Unpack all sub-dictionaries into the main row
    row.update(obj['biomarkers'])
    row.update(obj['treatment'])
    row.update(obj['temporal_alignment'])
    row.update(obj['outcomes'])
    
    # Add baseline volumetric states (Timepoint_1)
    if 'Timepoint_1' in obj['trajectories']:
        v1 = obj['trajectories']['Timepoint_1']
        for label, vol in v1.items():
            clean_name = label.split('(')[0].strip().replace(' ', '_')
            row[f"Baseline_{clean_name}"] = vol
    
    row['Total_Scans_Available'] = len(obj['timepoints'])
    master_rows.append(row)
    
    # --- B. Create Longitudinal Table (Dynamic States for Neural ODEs) ---
    for tp in obj['timepoints']:
        if tp in obj['trajectories']:
            # Find the actual day value from temporal_alignment for this timepoint
            day_val = obj['temporal_alignment'].get(tp)
            
            long_row = {
                'Patient_ID': pid,
                'Timepoint': tp,
                'Days_from_Diagnosis': day_val,
                # Include key priors for easy grouping/plotting
                'IDH1': obj['biomarkers'].get('IDH1'),
                'MGMT': obj['biomarkers'].get('MGMT')
            }
            # Add all volumes for this specific timepoint
            long_row.update(obj['trajectories'][tp])
            longitudinal_rows.append(long_row)

# 1. Create Master DataFrame (The 'Input-Output' Matrix)
master_df = pd.DataFrame(master_rows)

# 2. Create Longitudinal DataFrame (The 'Growth' Matrix)
long_df = pd.DataFrame(longitudinal_rows)

# --- CLEANING & FINAL SUMMARY ---
# Remove any accidental artifacts in column names
master_df.columns = [c.replace('_Vol', '').strip() for c in master_df.columns]

print(f"\n✅ SUCCESS: Integrated {len(master_df)} Patients.")
print(f"📐 Master Shape (Priors/Outcomes): {master_df.shape}")
print(f"📈 Longitudinal Shape (Growth States): {long_df.shape}")

# Display the first few rows of the VHT Master Data
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("\n--- MASTER VHT DATAFRAME SAMPLE ---")
print(master_df.head())

print("\n--- GROWTH TRAJECTORY SAMPLE ---")
print(long_df.head(10))

# OPTIONAL: Save to CSV for the next stage of your thesis
# master_df.to_csv("VHT_Master_Data.csv", index=False)
# long_df.to_csv("VHT_Longitudinal_Trajectories.csv", index=False)


📊 FLATTENING DATA FOR VHT MODELING...

✅ SUCCESS: Integrated 203 Patients.
📐 Master Shape (Priors/Outcomes): (203, 22)
📈 Longitudinal Shape (Growth States): (382, 10)

--- MASTER VHT DATAFRAME SAMPLE ---
       Patient_ID     Sex  Age  IDH1  MGMT  EGFR  1p19q RT_Dose  RT_Fractions Chemo  Surgery_Days  T1_Days  T2_Days  T3_Days  Progression  OS_Days  Baseline_Necrotic_Tumor_Core  Baseline_Tumor_Infiltration_and_Edema  Baseline_Enhancing_Tumor_Core  Baseline_Total_Tumorume  Total_Scans_Available  Baseline_Resection_Cavity
0  PatientID_0003  Female   57     0     4     2      0   60 Gy          30.0   Yes            -5     90.0    146.0      NaN            1      NaN                        6510.0                                44251.0                        33779.0                  84540.0                      3                        NaN
1  PatientID_0004  Female   67     0     4     2     10     NaN           NaN   NaN            -4     67.0      NaN      NaN            0      NaN     